# Vector Graph Memory Playground

This notebook demonstrates the basic setup of a PydanticAI agent with Qdrant and JanusGraph database tools.

## Setup and Imports

In [ ]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from gremlin_python.driver import client as gremlin_client
from pydantic_ai import Agent, RunContext
from pydantic import BaseModel

# Load environment variables
load_dotenv()

## Database Connections

In [ ]:
# Qdrant connection
qdrant_host = os.getenv('QDRANT_HOST', 'localhost')
qdrant_port = int(os.getenv('QDRANT_PORT', '6333'))

qdrant = QdrantClient(host=qdrant_host, port=qdrant_port)
print(f"Connected to Qdrant at {qdrant_host}:{qdrant_port}")

In [ ]:
# JanusGraph connection
janusgraph_host = os.getenv('JANUSGRAPH_HOST', 'localhost')
janusgraph_port = int(os.getenv('JANUSGRAPH_PORT', '8182'))

janus = gremlin_client.Client(
    f'ws://{janusgraph_host}:{janusgraph_port}/gremlin',
    'g'
)
print(f"Connected to JanusGraph at {janusgraph_host}:{janusgraph_port}")

## Database Context

In [ ]:
class DatabaseContext(BaseModel):
    """Context containing database connections for the agent."""
    qdrant_client: QdrantClient
    janus_client: gremlin_client.Client
    
    class Config:
        arbitrary_types_allowed = True

## Agent Tools

In [ ]:
# Create the agent with database context
agent = Agent(
    'openai:gpt-4',
    deps_type=DatabaseContext,
    system_prompt="You are a helpful assistant with access to a vector-graph memory system."
)

@agent.tool
async def search_vector_memory(ctx: RunContext[DatabaseContext], query: str, limit: int = 5) -> str:
    """
    Search the vector database for semantically similar content.
    
    Args:
        query: The search query text
        limit: Maximum number of results to return
    
    Returns:
        Search results as a formatted string
    """
    # TODO: Implement actual vector search with embeddings
    # For now, return a placeholder
    return f"Vector search for '{query}' (limit: {limit}) - Not yet implemented"

@agent.tool
async def query_graph(ctx: RunContext[DatabaseContext], gremlin_query: str) -> str:
    """
    Execute a Gremlin query on the graph database.
    
    Args:
        gremlin_query: The Gremlin query to execute
    
    Returns:
        Query results as a formatted string
    """
    try:
        result = ctx.deps.janus_client.submit(gremlin_query).all().result()
        return f"Graph query results: {result}"
    except Exception as e:
        return f"Error executing graph query: {str(e)}"

@agent.tool
async def add_to_memory(ctx: RunContext[DatabaseContext], content: str, entity_type: str) -> str:
    """
    Add new content to both vector and graph databases.
    
    Args:
        content: The content to store
        entity_type: The type of entity (e.g., 'job', 'company', 'person')
    
    Returns:
        Confirmation message
    """
    # TODO: Implement actual storage
    # 1. Generate embedding for content
    # 2. Store in Qdrant
    # 3. Create node in JanusGraph
    # 4. Link vector ID to graph node
    return f"Added {entity_type} to memory: '{content}' - Not yet implemented"

print("Agent tools registered:")
print("- search_vector_memory: Search for semantically similar content")
print("- query_graph: Execute Gremlin queries on the graph")
print("- add_to_memory: Store new entities in the memory system")

## Test the Agent

In [ ]:
# Create database context
db_context = DatabaseContext(
    qdrant_client=qdrant,
    janus_client=janus
)

# Run a simple query
result = await agent.run(
    "What tools do you have available?",
    deps=db_context
)

print(result.data)

## Example: Job Search Use Case

In [ ]:
# Example: Add a job to memory
result = await agent.run(
    "Add a software engineer position at Google to my memory",
    deps=db_context
)

print(result.data)

In [ ]:
# Example: Search for jobs
result = await agent.run(
    "Search for engineering positions in my memory",
    deps=db_context
)

print(result.data)

## Cleanup

In [ ]:
# Close connections
janus.close()
print("Database connections closed")